In [ ]:
import cda2
import json
import re
import pyspark.sql.types as T
import pyspark.sql.functions as F
#import datetime
import time

from datetime import datetime, timedelta
from timeit import default_timer as timer
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [ ]:
api = cda2.Api(timeout=120)

Set configuration parameters to better optimize queries.

In [ ]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "16g",
    "spark.executor.memoryOverhead": "16g",
    "spark.reducer.maxReqsInFlight": "1",
    "spark.shuffle.io.retryWait": "60s",
    "spark.shuffle.io.maxRetries": "10",
}

# config = {
#     # "spark.sql.adaptive.enabled": "true",
#     # "spark.sql.adaptive.coalescePartitions.enabled": "true",
#     # "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
#     # "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
#     "spark.executor.memory": "16g",
#     "spark.executor.memoryOverhead": "16g",
# }

config = {
    "spark.hadoop.fs.s3a.fast.upload.buffer": "bytebuffer",
}


Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [ ]:
#api.start_spark(n_executors=400, config=config)
api.start_spark(n_executors=400)

In [ ]:
api.spark.sparkContext.setLogLevel("ERROR")

In [ ]:
year0 = "2026" # specify the year as a yyyy string

In [ ]:
# year0 = "2025"
year1 = str(int(year0) + 1)

In [ ]:
datelist = [
    year0 + "0101",
    year0 + "0201",
    year0 + "0301",
    year0 + "0401",
    year0 + "0501",
    year0 + "0601",
    year0 + "0701",
    year0 + "0801",
    year0 + "0901",
    year0 + "1001",
    year0 + "1101",
    year0 + "1201",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0401",
    year0 + "0701",
    year0 + "1001",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0301",
    year0 + "0501",
    year0 + "0701",
    year0 + "0901",
    year0 + "1101",
    year1 + "0101",
]

datelist = [
    year0 + "0901",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0501",
    year0 + "0901",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0401",
    year0 + "0701",
    year0 + "1001",
    year1 + "0101",
]


In [ ]:
i = 0
dates = {"start_date": datelist[i], "end_date": datelist[i+1]}

In [ ]:
print("starting: ", dates)

In [ ]:
airports = [
    "KADW", "KATL", "KBOS", "KBWI", "KCLT", "KDCA", "KDEN", "KDFW", "KDTW",
    "KEWR", "KFLL", "KIAD", "KIAH", "KJFK", "KLAS", "KLAX", "KLGA", "KMCO",
    "KMDW", "KMEM", "KMIA", "KMSP", "KORD", "KPHL", "KPHX", "KSAN", "KSDF",
    "KSEA", "KSFO", "KSLC", "KTPA", "PANC", "PHNL",
]

airports = [
    "KMEM",
]

In [ ]:
#if do_planned_routes:

print("   retrieving planned_routes: ", datetime.now())

## get fix_key and associated icao_region
df_arincfix = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .select(
        F.col("primary_key").alias("fix_key"),
        F.col("identification.icao_region").alias("icao_region"),
    )
)

## get the route expansion and join with arincfix to get the icao_region for each fix
df_plannedroutes_raw = (
    api.dataframe("PlannedRoute", **dates, partition_filters=api.custom_partitions("ASSOCIATED"), metadata=True)
    .select(
        "track_key",
        "route",
        F.col("total_route_length").alias("route_length_nm"),
        F.col("labels.is_lfpd").alias("is_lfpd"),
        F.explode("legs").alias("leg"),
        F.col("leg.sequence_number").alias("seqnum"),
        F.col("leg.path_terminator.fix_name").alias("fix_name"),
        F.col("leg.path_terminator.fix_key").alias("fix_key"),
    )
    .filter(F.col("track_key") == '4btEpPWoY6M--BC')
    .filter(F.col("is_lfpd") == 'true')
    .withColumn("route_length_nm", F.round("route_length_nm", 1))
    .join(df_arincfix, on=["fix_key"], how="inner")
    .withColumn("fix_name_uniq", F.concat("fix_name", F.lit("("), "icao_region", F.lit(")")))
    .drop("legs", "leg", "is_lfpd", "icao_region", "fix_key")
)

## remove any duplicate fixes in the route expansion
window = Window.partitionBy("track_key", "fix_name").orderBy(col("seqnum").asc())
df_plannedroute_raw_dedup = (
    df_plannedroutes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
    .orderBy("track_key", "seqnum")
)

## make plannedroutes dataframe with route expansion strings fix_name_expansion and fix_name_uniq_expansion 
df_plannedroutes = (
    df_plannedroute_raw_dedup
        .groupby("track_key", "route_length_nm", "route")
        .agg(F.sort_array(F.collect_list(F.struct("seqnum", "fix_name"))).alias("fix_name_list"), F.sort_array(F.collect_list(F.struct("seqnum", "fix_name_uniq"))).alias("fix_name_uniq_list"))
        .withColumn("fix_name_expansion", F.concat_ws(".", F.col("fix_name_list.fix_name")))
        .withColumn("fix_name_uniq_expansion", F.concat_ws(".", F.col("fix_name_uniq_list.fix_name_uniq")))
        .drop("fix_name_list", "fix_name_uniq_list")
)

In [ ]:
## get the flightplans for tracks to and from airports of interest

#if do_flightplans:

print("   retrieving flightplans: ", datetime.now())

df_flightplanseries = (
    api.dataframe("FlightplanSeries", **dates, metadata=True)
    .select(
        "track_key",
        "metadata.effective_start_date",
        "metadata.effective_end_date",
        "callsign",
        "aircraft_type",
        "mode_s_code",
        F.col("initial_departure_aerodrome").alias("orig"),
        F.col("final_destination_aerodrome").alias("dest"),
    )
    .withColumn('start_date',F.from_unixtime(F.col('effective_start_date')/1000, 'yyyyMMdd'))
    .withColumn('end_date',F.from_unixtime(F.col('effective_end_date')/1000, 'yyyyMMdd'))
    .withColumn("month", F.substring("start_date", 5, 2))
    .withColumn("carrier", F.regexp_extract("callsign", "^([A-Za-z]+)\d*[A-Za-z]*$", 1))
    .filter(F.col("orig") != F.col("dest"))
    .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
    .drop("effective_start_date", "effective_end_date")
)

In [ ]:
print("   joining df_plannedroutes to df_flightplanseries: ", datetime.now())

In [ ]:
df_joined = df_plannedroutes.join(df_flightplanseries, on=["track_key"], how="inner")

In [ ]:
df_plannedroutes_raw = (
    api.dataframe("PlannedRoute", **dates, partition_filters=api.custom_partitions("ASSOCIATED"), metadata=True)
    .filter(F.col("track_key") == '4btEpPWoY6M--BC')
)

In [ ]:
df_plannedroutes_raw.select("legs").show(truncate=False)

In [ ]:
df_plannedroutes.show(truncate=False)

In [ ]:
df_joined.show(truncate=False)